In [ ]:
import torch
import torch.nn as nn 

from torch.utils.data import TensorDataset, DataLoader


### Generated data

In [ ]:
#100 samples, 3 features
X = torch.rand(100,3)
y = torch.randint(0,2,(100,1)) #generate fake binary class labels


train_dataset = TensorDataset(X,y) #wrap the tensors into a dataset
valid_dataset = TensorDataset(X,y) #wrap the tensors into a dataset


In [ ]:

#create a data loader for batching and shuffling
train_loader = DataLoader(
    dataset, 
    batch_size=32, 
    shuffle=True)  #shuffle means shuffling order each epoch

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32, 
    shuffle=False
)


### Training mode

In [ ]:
model = nn.Linear(3,2) #3 input features, 2 possible output classes

criterion = nn.CrossEntropyLoss() #loss function for multi-class classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) #optimizer for updating model parameters



In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
# we want all computations to be done on the same device (CPU, GPU, or MPS)
model = model.to(device)

model.train() #activates traiing mode (enable dropout, batchnorm)

In [ ]:
for epoch in range(10):
    for X_batch, y_batch in train_loader:



        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch) #forward pass
        loss = criterion(outputs, y_batch)

        loss.backward() #backprop: compute gradients
        optimizer.step() #update model param

### Eval mode

In [ ]:

model.eval()
with torch.no_grad(): #disable gradient computation for validation
    for X_batch, y_batch in valid_loader: 
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch) #forward pass 
        loss = criterion(outputs, y_batch) 


        #class predictions: gives idx of the class w highest logit
        predicted = outputs.argmax(dim=1) 
        #accuracy 
        total = y_batch.size(0)
        correct = (predicted == y_batch).sum().item()
        accuracy = correct / total



